# Lab 1: Tokenisation and embeddings

In this lab, you will build an understanding of how text can be transformed into representations that computers can process and learn from. Specifically, you will explore two key concepts: *tokenisation* and *embeddings*. Tokenisation splits text into smaller units such as words, subwords, or characters. Embeddings are dense, fixed-size vector representations of tokens in a continuous space.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Part 1: Tokenisation

In the first part of the lab, you will code and analyse a tokeniser based on the Byte Pair Encoding (BPE) algorithm.

### Utility functions

The BPE tokeniser transforms text into a list of integers representing tokens. As a warm-up, you will implement two utility functions on such lists. To simplify things, we define a shorthand for the type of pairs of integers:

In [17]:
type Pair = tuple[int, int]
#print((1, 2))

#### 🧩 Task 1.01: Counting pairs

Write a function that counts all occurrences of pairs of consecutive token IDs in a given list. The function should return a dictionary that maps each pair to its count. Skip counts that are zero.

ANSWER: the code below

In [18]:
def count(ids: list[int]) -> dict[Pair, int]:
  counts = {}
  for i in range(len(ids) - 1):
      current_pair = (ids[i], ids[i+1])
      if current_pair in counts:
        counts[current_pair] += 1
      else:
        counts[current_pair] = 1
  return counts

test_1 = [1, 2, 3, 1, 2]
test_2 = [1, 2, 3, 4]
test_3 = [3]

print(count(test_1))
print(count(test_2))
print(count(test_3))

{(1, 2): 2, (2, 3): 1, (3, 1): 1}
{(1, 2): 1, (2, 3): 1, (3, 4): 1}
{}


#### 🧩 Task 1.02: Replacing pairs

Write a function that traverses a list of token IDs from left to right and replaces all occurrences of a specified pair of consecutive IDs by a new ID. The function should return the modified list.

ANSWER: the code below

In [19]:
def replace(ids: list[int], pair: Pair, new_id: int) -> list[int]:
  new_ids = []
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
        new_ids.append(new_id)
        i += 2
    else:
        new_ids.append(ids[i])
        i += 1
  return new_ids

test_1 = ([1, 2, 3, 1, 2], (1, 2), 99)
test_2 = ([1, 2, 3, 4], (4, 5), 99)
test_3 = ([3], (1, 2), 99)
print(replace(*test_1))
print(replace(*test_2))
print(replace(*test_3))

[99, 3, 99]
[1, 2, 3, 4]
[3]


### Encoding and decoding

The next cell contains the core code for the tokeniser in the form of a class `Tokenizer`. This class implements two methods: `encode()` converts an input text to a list of token IDs by exhaustively applying rules for merging pairs of consecutive IDs (stored in the dictionary `self.merges`), and `decode()` reverses this process.

**Note that the set of merge rules is initially empty; you will add rules in Task&nbsp;1.04.**

In [20]:
class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(2**8)}

    step = 1
    def encode(self, text):
        #print (f"--- Encoding text: {text} ---")
        ids = list(text.encode("utf-8"))
        #print(f"Initial ids: {ids}")
        while True:
            counts = count(ids)
            mergeable_pairs = counts.keys() & self.merges.keys()
            #print(f"Mergeable pairs: {mergeable_pairs}")
            if len(mergeable_pairs) == 0:
                break
            to_merge = min(mergeable_pairs, key=self.merges.get)  # type: ignore
            #print(f"Step {self.step}: Merging pair {to_merge} into id {self.merges[to_merge]}")
            self.step += 1
            ids = replace(ids, to_merge, self.merges[to_merge])
            #print(f"Ids after merge: {ids}")
        #print(f"Final ids: {ids}")
        return ids

    def decode(self, ids):
        return b"".join((self.vocab[i] for i in ids)).decode("utf-8")

#### 🎓 Task 1.03: Encoding and decoding

Explain how the code implements the BPE algorithm. Use the following steps to check your understanding:

**Step&nbsp;1.** Annotate the attributes and methods of the `Tokenizer` class with their Python types. In particular, what is the type of `self.merges`? Use the `Pair` shorthand. What does a merge rule look like?

ANSWER: The type is a dict[tuple[int, int], int]. It matches a tuple of two integers (the pair to become merged) to a single integer (the new ID). 
A single entry in self.meges could look like: {(100, 101): 256} which means, whenever you see the token ID 100 followed by token ID 101, replace it with new token ID 256. 

**Step&nbsp;2.** Explain how the implementation chooses which merge rule to apply. Provide an example that illustrates the logic. Construct the example such that you get a different result when you use `max()` instead of `min()`.

ANSWER: The implementation selects which merge rule to apply based on the value of the new ID associated with the pair in self.mergs. 
The new token IDs are assigned sequentially (e.g, 256 then 257 and so on). Thefore the lowest ID corresponds the the earliest rule learned during training. For example if self.merges contains [10, 20, 30] there are two overlapping pairs -> (10, 20) and (20, 30). Lets assume that the merge rule says -> (10, 20): 256 (learned firs)
                            (20, 30): 257 (learned second)
then min() selects 256 and the result becomes -> [256, 30]
What if it was max()? -> We wuld instead had -> [10, 257]
Using min() ensures that we prioritize the ealiear rule. 

### Training a tokeniser

Upon initialisation, a tokeniser has an empty set of merge rules. Your next task is to complete the BPE algorithm and write code to learn these merge rules from a text.

#### 🎓 Task 1.04: Training a tokeniser

Write a function that induces a BPE tokeniser from a given text. The function should take the text (a string) and a target vocabulary size as input and return the trained tokeniser.

ANSWER: The code below

In [21]:
# def from_text(text: open, vocab_size: int) -> Tokenizer:
#     tok = Tokenizer()
#     ids = list(text.encode("utf-8"))
#     print(f"list of ideas: {ids}")
#     num_merges = vocab_size - 256

#     for i in range(num_merges):
#         stats = count(ids)
#         print(f"stats: {stats}")
#         if not stats:
#             break
#         pair = max(stats, key=stats.get)
#         print(f"paris:' {pair}")
#         idx = 256 + i
#         ids = replace(ids, pair, idx)

#         tok.merges[pair] = idx

#         tok.vocab[idx] = tok.vocab[pair[0]] + tok.vocab[pair[1]]

#     return tok

# sample_text = "low low low lower newest"
# trained_tok = from_text(sample_text, vocab_size=260)

# print("\n--- Final Learned Merges ---")
# print(trained_tok.merges)
# print("\n--- Testing the Learned Tokenizer ---")
# encoded = trained_tok.encode("low")
# print(f"Encoding 'low': {encoded}")


#------
# def load_tokenizer(path: str) -> Tokenizer:
#     tok = Tokenizer()
#     with open(path, "r", encoding="utf-8") as f:
#         for i, line in enumerate(f):
#             parts = list(map(int, line.strip().split()))
#             #print(f"parts: {parts}")
#             pair = (parts[0], parts[1])
#             #print(f"pair: {pair}")
#             idx = 256 + i
#             tok.merges[pair] = idx
#             tok.vocab[idx] = tok.vocab[pair[0]] + tok.vocab[pair[1]]
#     return tok

# with open( "wiki-en-1m.txt", "r", encoding="utf-8") as f:
#     text_en = f.read()

# print(f"Loaded English text of length {len(text_en)}")
# #print (f"snippet: {text_en[:100]}")

# Tokenizer_en = load_tokenizer("wiki-en-1m.tok")

# print (f"--- Loaded Tokenizer on English Text with {len(Tokenizer_en.merges)} merges ---")

# # Let's encode the first 100 characters of the text
# sample = text_en #[:5]
# encoded_ids = Tokenizer_en.encode(sample)
# decoded_text = Tokenizer_en.decode(encoded_ids)
# #print(f"Encoded IDs: {encoded_ids}")

# # print("\n--- Verification ---")
# # print(f"Original: '{sample}'")
# # print(f"Encoded:  {encoded_ids}")
# # print(f"Decoded:  '{decoded_text}'")

# # Check compression
# original_size = len(sample.encode('utf-8'))
# print(f"Original Size (bytes): {original_size}")
# compressed_size = len(encoded_ids)
# print(f"Compressed Size (ids): {compressed_size}")

# print(f"Compression Ratio: {original_size / compressed_size:.2f}X")



def from_text(text: str, vocab_size: int) -> Tokenizer:
    tok = Tokenizer()
    # print(f" length of the text: {len(text)}")
    ids = list(text.encode('utf-8'))
    #print(f"Initial tokens (Bytes): {len(ids)}")

    idx = 256
    while len(tok.vocab) < vocab_size:
        counts = count(ids)
        if not counts:
            break

        most_freq_pair = max(counts, key=counts.get)
        occurrences = counts[most_freq_pair]
        # print(f"\n[Step {idx}]")
        # print(f"  Most Frequent Pair: {most_freq_pair} appeared {occurrences} times")

        tok.merges[most_freq_pair] = idx

        p0, p1 = most_freq_pair
        new_token_bytes = tok.vocab[p0] + tok.vocab[p1]
        tok.vocab[idx] = new_token_bytes
        # print(f" Created new token id {idx}: {new_token_bytes}")

        ids = replace(ids, most_freq_pair, idx)
        # print(f" text length reduced to: {len(ids)}")
        idx+=1
    # print(f"\n--- Training Complete. Final Vocab Size: {len(tok.vocab)} ---")
    return tok


To help you test your implementation, we provide three text files together with tokenisers trained on these files. Each text file contains the first 1&nbsp;million Unicode characters in a language-specific Wikipedia:

| Text file | Tokeniser file | Wikipedia |
|---|---|---|
| `wiki-en-1m.txt` | `wiki-en-1m.tok` | [Simple English](https://simple.wikipedia.org/) |
| `wiki-is-1m.txt` | `wiki-is-1m.tok` | [Icelandic](https://is.wikipedia.org/) |
| `wiki-sv-1m.txt` | `wiki-sv-1m.tok` | [Swedish](https://sv.wikipedia.org/) |

A tokeniser file consists of lines specifying merge rules. For example, the first line in the tokeniser file for Swedish is `101 114`, which expresses that this rule combines the token with ID 101 (`e`) and the token with ID 114 (`r`). The ID of the new token (`er`) is 256 plus the (zero-indexed) line number on which the rule is found. The following code saves a `Tokenizer` to a file with this format:

In [22]:
def save(tokenizer: Tokenizer, filename: str) -> None:
    with open(filename, "w") as f:
        for fst, snd in tokenizer.merges:
            print(f"{fst} {snd}", file=f)

In [23]:
def read_files(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()

file_content = read_files("wiki-en-1m.txt")
tokenizer = from_text(file_content, vocab_size=1024)
save(tokenizer, "new-wiki-en-1m.tok")

#for iscland language
file_content_is = read_files("wiki-is-1m.txt")
# tokenizer_is = from_text(file_content_is, vocab_size=1024)
# save(tokenizer_is, "new-wiki-is-1m.tok")

In [24]:
print(f"--English Text--")
eng_tok = tokenizer.encode(file_content)
num_tok_eng = len(eng_tok)
num_char_eng = len(file_content)
print(f"Total Eglish Characters: {num_char_eng}")
print(f"Total English Tokens: {num_tok_eng}")

print(f"\n--Icelandic Text--")
is_tok = tokenizer.encode(file_content_is)
num_tok_is = len(is_tok)
num_char_is = len(file_content_is)
print(f"Total Icelandic Characters: {num_char_is}") 
print(f"Total Icelandic Tokens: {num_tok_is}")

--English Text--
Total Eglish Characters: 1000000
Total English Tokens: 379779

--Icelandic Text--
Total Icelandic Characters: 1000000
Total Icelandic Tokens: 754866


In [25]:
!diff wiki-en-1m.tok new-wiki-en-1m.tok
# !diff wiki-is-1m.tok my-wiki-is-1m.tok
# !diff wiki-sv-1m.tok my-wiki-sv-1m.tok

To test your code, compare your saved tokeniser to the provided tokeniser using the `diff` command line tool.

**Note that training a tokeniser can take a few minutes.**

### Tokenisation quirks

The tokeniser is a key component of language models, as it defines the minimal chunks of text the model can “see” and work with. As you will see in this section, tokenisation is also responsible for several deficiencies and unexpected behaviours of language models.

One helpful tool for experimenting with tokenisers in language models is the web app [Tiktokenizer](https://tiktokenizer.vercel.app/). This app lets you play around with, among others, [`o200k_base`](https://tiktokenizer.vercel.app/?model=o200k_base), the tokeniser used in ChatGPT&nbsp;5.2. You can also use OpenAI’s own [Tokenizer](https://platform.openai.com/tokenizer) page.

#### 🎓 Task 1.05: Tokenisation quirks

Prompt [ChatGPT](https://chatgpt.com/) to reverse the letters in the following words:

```
creativecommons
MERCHANTABILITY
NSNotification
authentication
```

How many of these words come out right? What happens when you modify the prompt and explicitly disable “thinking” and external tools? What could be the problem when words come out wrong? Generate ideas by inspecting the words in Tiktokenizer. Try to come up with other prompts that illustrate problems related to tokenisation.

ANSWER: All of them were right by the "thinking" mode actvie. Three of them were right by "thinking" mode deactive. 
The problem is the inspection with the Tiktokinizer. 
Some other ideas: "Strawberry" -> one of the 'r' could not be seen.
                  "Aibohphobia" -> the model could not compare the first and the last part of the word. 

### Tokenisation and multi-linguality

Many NLP systems and the tokenisers used with them are primarily trained on English data. In the next task, you will reflect on the effect this has when they are used to process non-English data.

The *context length* of a language model is the maximum number of preceding tokens the model can condition on when predicting the next token. This number is fixed and cannot be changed after training the model. For example, the context length of GPT-2 ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)) is 1,024. 

While the context length of a language model is fixed, the amount of information that can be squeezed into this context length will depend on the tokeniser. Informally speaking, a model that needs more tokens to represent a given text cannot extract as much information from that text as one that needs fewer tokens.

#### 🎓 Task 1.06: Tokenisation and multi-linguality

Train a tokeniser on the English text file from Task&nbsp;1.04 (or use the provided one) and apply it to the same text. How many tokens does it split the text into? Based on this, what is the expected number of Unicode characters of English text that can be fit into the GPT-2 context window?

What do the numbers look like if you apply the English tokeniser to the Icelandic text instead? How do you explain the differences?

ANSWER: check the uputput in the code snippets above. 

Interpreting the expected number of Unicode characters as a measure of representation efficiency, what do your results tell you about the efficiency of a language model primarily trained on English data when it is used to process non-English data? Relate these findings to the issue of tokeniser fairness.

## Part 2: Embeddings

In the second part of the lab, you will explore embeddings. An embedding layer is a network component that assigns each item in a finite set of elements (often called a *vocabulary*) a fixed-size vector. At first, these vectors are filled with random values, but during training, they are adjusted to suit the task at hand.

### Bag-of-words classifier

To help you build an intuition for embeddings and the vector representations learned by them, we will use a simple bag-of-words text classifier. The core part of this classifier only takes a few lines of code:

In [26]:
import torch.nn as nn


class Classifier(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))

#### 🧩 Task 1.07: Bag-of-words classifier

Explain how the bag-of-words classifier works. How does the code match the diagram you saw in the lectures? Why is there only one `nn.Embedding`, while the diagram shows three embedding layers? What does the keyword argument `dim=-2` do? 
BoW is a simple text representation method in NLP that reats an unordered collection of words by focusing on word frequency while disregarding grammar, order and context. It converts the text into numbers which are used by ML models. 

The answers:

BoW is a simple text representation method in NLP that treats an unordered collection of words by focusing on word frequency while disregarding grammar, order and context. It converts the text into numbers which are used by ML models. In our case when we are done with tokenization stage and get the IDs, which are used as an input (x in the code) to calculate an average list of it. This will later be used in neural network layer. 

The code matches the diagram well, becuase the digram shows that the embedding layer is between the tokenization and neural network layesr and takes the token IDs from the tokenization layer as input to convert to flots tensors and calculate the mean of it. 

Regardless how many embedding layers are shown in the diagram, they are treated by the same matematheca function, The diagram just shows that the function is applied to three different words not three different function (nn.Embedding).

It tells PyTorch to average all the words in the sentnce together into one single summary


### Dataset

You will apply the classifier to a small dataset with Amazon customer reviews. This dataset is taken from [a much larger dataset](https://www.cs.jhu.edu/~mdredze/datasets/sentiment/) first described by [Blitzer et al. (2007)](https://aclanthology.org/P07-1056/).

The dataset contains whitespace-tokenised product reviews from two categories: cameras (`camera`) and music (`music`). Each review is additionally annotated for sentiment towards the product at hand: negative (`neg`) or positive (`pos`). The category and sentiment labels are prepended to the review. As an example, here is the first review from the training data:

```
music neg oh man , this sucks really bad . good thing nu-metal is dead . thrash metal is real metal , this is for posers
```

The next cell contains a custom [`Dataset`](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) class for the review dataset. To initialise an instance of this class, you specify the name of the file containing the reviews you want to load (`filename`) and which of the two labels you want to use (`label`): product category (0) or sentiment (1).

In [27]:
from torch.utils.data import Dataset


class ReviewDataset(Dataset):
    def __init__(self, filename: str, label: int = 0) -> None:
        print(f"--- [STEP 1] LOADING DATA FROM {filename} ---")  ####
        with open(filename) as f:
            tokenized_lines = [line.split() for line in f] [:5] ### only load first 5 lines for testing
        print(f"--- [STEP 2] LOADED {len(tokenized_lines)} LINES.")
        print("splited data: {tokenized_lines}")
        self.items = [(tokens[2:], tokens[label]) for tokens in tokenized_lines]

        # PRINT TRACE
        for i, item in enumerate(self.items):  # Print first 2 items for trace
            print(f"  Raw Item {i}: Label='{item[1]}', Text={item[0][:5]}...")

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[list[str], str]:
        return self.items[idx]

### Vectoriser

To feed a review into the bag-of-words classifier, you first need to turn it into a vector of token IDs. Likewise, you need to convert the label (product category or sentiment) into an integer. The next cell contains a partially completed `ReviewVectoriser` class that handles this transformation.

In [28]:
from collections import Counter

import torch

# Type abbreviation for review–label pairs
type Item = tuple[list[str], str]



class ReviewVectorizer:
    PAD = "[PAD]"
    UNK = "[UNK]"

    def __init__(self, dataset: ReviewDataset, n_vocab: int = 1024) -> None:
        # Unzip the dataset into reviews and labels
        reviews, labels = zip(*dataset)
        print(f"reviews: {reviews}, \nlabels: {labels}")

        # Count the tokens and get the most common ones
        # counter = Counter(t for r in reviews for t in r)
        # most_common = [t for t, _ in counter.most_common(n_vocab - 2)]
        # print(f"Most_Common Tokens: {most_common} ")

        # Create the token-to-index and label-to-index mappings

        # self.t2i = {t: i for i, t in enumerate([self.PAD, self.UNK] + most_common)}
        # self.l2i = {l: i for i, l in enumerate(sorted(set(labels)))}

        print("\n--- BUILDING LABEL MAPPING (l2i) ---")
        unique_labels = set(labels)
        print(f" A. set (Unique): {unique_labels}")
        sorted_labels = sorted(unique_labels)
        print(f" B. Sorted List: {sorted_labels}")
        self.l2i = {l: i for i, l in enumerate(sorted_labels)}
        print(f" C. Final Dict: {self.l2i}")

        # Step A: count
        print("\n --- BUILDING TOKEN MAPPING (t2i) ----")
        counter = Counter(t for r in reviews for t in r)
        most_common = [t for t, _ in counter.most_common(n_vocab - 2)]
        print(f"   A. Most Common Words found: {most_common}")

        #Step B: Add special tokens
        special_tokens = [self.PAD, self.UNK]
        full_vocab_list = special_tokens + most_common
        print(f"   B. Full List (Special + Common): {full_vocab_list}")
        self.t2i = {t: i for i, t in enumerate(full_vocab_list)}

        print(f"   C. Final Token Dict (First 10):")   
        for k, v in list(self.t2i.items())[:10]:
            print(f"      '{k}': {v}")



    # def __call__(self, items: list[Item]) -> tuple[torch.Tensor, torch.Tensor]:
    #     # TODO: Complete the implementation of this method
    #     print(f"\n--- [STEP 3] VECTORIZING BATCH ---")
    #     reviews, labels = zip(*items)
    #     print(f"  Input Labels (String): {labels}")
    #     label_indices = [self.l2i[label] for label in labels]
    #     print(f"  Input Labels (Int):    {label_indices}")

    #     ukn_ideas = self.t2i[self.UNK]
    #     review_indices = [
    #         [self.t2i.get(token, ukn_ideas) for token in review]
    #         for review in reviews
    #     ]
    #     print(f"  Jagged IDs (Before Pad): {review_indices}")

    #     max_len = max(len(r) for r in review_indices)
    #     padd_idea = self.t2i[self.PAD]
    #     padded_reviews = []

    #     for i in review_indices:
    #         num_padding = max_len - len(i)
    #         padded_reviews.append(i + [padd_idea] * num_padding)

    #     # PRINT TRACE
    #     print(f"  Rectangular IDs (After Pad): {padded_reviews}")
    #     return torch.tensor(padded_reviews, dtype=torch.long), torch.tensor(label_indices, dtype=torch.long)


    def __call__(self, items: list[Item]) -> tuple[torch.Tensor, torch.Tensor]:
            print(f"\n--- [VECTORIZER START] Processing batch of {len(items)} items ---")

            # 1. UNZIP: Separate questions (reviews) from answers (labels)
            reviews, labels = zip(*items)

            # 2. CONVERT LABELS: String -> Int
            # We look up 'music' (1) or 'camera' (0) in the l2i dictionary
            label_ids = [self.l2i[l] for l in labels]

            print(f"  A. Labels: {labels}")
            print(f"  B. Label IDs: {label_ids}")

            # 3. CONVERT TOKENS: String -> Int (Handling UNK)
            # We prepare the fallback ID for unknown words
            unk_id = self.t2i[self.UNK]

            review_ids_jagged = []
            for review in reviews:
                # Try to find the word. If not found, return unk_id (1)
                ids = [self.t2i.get(token, unk_id) for token in review]
                review_ids_jagged.append(ids)

            print(f"  C. Jagged IDs (First 2): {review_ids_jagged[:2]}...")

            # 4. PADDING: Make them a Rectangle
            # Find the length of the longest review in THIS batch
            max_len = max(len(r) for r in review_ids_jagged)
            pad_id = self.t2i[self.PAD]
            #print(f"Pad_IDs:  {pad_id}")

            print(f"  D. Max Length in batch: {max_len}")

            padded_reviews = []
            for r in review_ids_jagged:
                # Calculate how many 0s we need
                num_pads = max_len - len(r)
                # Add them to the end
                padded_reviews.append(r + [pad_id] * num_pads)
            print(f"Padded_review:  {padded_reviews}")

            # 5. TENSOR CONVERSION
            # We must use dtype=torch.long (Integers) for Embedding layers
            x_tensor = torch.tensor(padded_reviews, dtype=torch.long) # padded_review converted to Pythorch Tensor, which is as input for neural networks
            y_tensor = torch.tensor(label_ids, dtype=torch.long) # the target vectors, a list of labels converted to Tensor

            print(f"  E. Final Tensor Shape: {x_tensor.shape}")

            return x_tensor, y_tensor


A `ReviewVectoriser` maps tokens and labels to IDs using two Python dictionaries. These dictionaries are set up when the vectoriser is initialised and queried when the vectoriser is called on a batch of review–label pairs. They include IDs for two special tokens:

`[PAD]` (Padding): Reviews can have different lengths, but PyTorch requires all vectors in a batch to be the same size. To handle this, the vectoriser adds `[PAD]` tokens to the end of shorter reviews so they match the length of the longest review in the batch.

`[UNK]` (Unknown): If a review contains a token that is not in the token-to-ID dictionary, the vectoriser assigns it the ID of the `[UNK]` token instead of a regular ID.

#### 🎓 Task 1.08: Vectoriser

Explain and complete the code of the vectoriser. Follow these steps:

**Step&nbsp;1.** Explain how unzipping works. What are the types of `reviews` and `labels`?

**Step&nbsp;2.** Explain how the token-to-ID and label-to-ID mappings are constructed. How does the `most_common()` method deal with elements that occur equally often?

**Step&nbsp;3.** Complete the implementation of the `__call__()` method. This method should convert a list of $m$ review–label pairs into a pair $(X, y)$ where $X$ is a matrix containing the vectors with token IDs for the reviews, and $y$ is a vector containing the IDs of the corresponding labels.

### Training the classifier

With the vectoriser completed, you are ready to train a classifier. More specifically, you can train two separate classifiers: one to predict the product category of a review, and one to predict the sentiment. The next cell contains a simple training loop that you can adapt for this purpose.

In [29]:
import torch.nn.functional as F


def train():
    print("--- [START] Initializing Data and Model ---") # <--- ADDED
    dataset = ReviewDataset("reviews-train.txt", label=0) 
    print(f"All Data: {dataset.items}") # <--- ADDED
    processor = ReviewVectorizer(dataset, 1024)
    print(f"Labels Mapping: {processor.l2i}")


    # 4. PRINT THE TENSORS
    x_tensor, y_tensor = processor(dataset.items)
    print("\n========== Y TENSOR (The Labels) ==========")
    print(f"Shape: {y_tensor.shape}")
    print(y_tensor)

    print("\n========== X TENSOR (The Reviews) ==========")
    print(f"Shape: {x_tensor.shape}")
    # We print the whole thing. PyTorch will show corners and hide the middle if it's huge.
    print(x_tensor)

    # OPTIONAL: See a specific row nicely formatted (e.g., the short one)
    print("\n--- Zooming in on the 4th Review (Index 3) ---")
    print(x_tensor[3])


    model = Classifier(1024, 64, len(processor.l2i))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=16,
        shuffle=True,
        collate_fn=processor,
    )
    for epoch in range(10):
        model.train()
        running_loss = 0
        print(f"\n--- Epoch {epoch} Start ---") # <--- ADDED
        #for bx, by in data_loader:
        for batch_idx, (bx, by) in enumerate(data_loader):
            print(f"  [Batch {batch_idx}] Processing {len(bx)} reviews...")
            optimizer.zero_grad()
            output = model(bx)
            loss = F.cross_entropy(output, by)

            # LET'S SEE THE PREDICTION # <--- ADDED
            winners = torch.argmax(output, dim=1) 
            print(f"    Predictions: {winners.tolist()}") # <--- ADDED
            print(f"    True Labels: {by.tolist()}")      # <--- ADDED
            print(f"    Batch Loss:  {loss.item():.4f}")  # <--- ADDED

            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch}, Avg Loss: {running_loss / len(data_loader):.4f}")
    return processor, model

#### 🎓 Task 1.09: Training loop

Explain the training loop. Follow these steps:

**Step&nbsp;1.** Go through the training loop line-by-line and add comments where you find it suitable. Your comments should be detailed enough for you to explain the main steps of the loop.

**Step&nbsp;2.** The training loop contains various hard-coded values like filename, learning rate, batch size, and epoch count. This makes the code less flexible. Revise the code so that you can specify these values using keyword arguments. Use the concrete values from the code as defaults.

#### 🧩 Task 1.10: Training the classifier

Adapt the next cell to train the classifier for the two prediction tasks. Based on the loss values, which task appears to be the harder one? What is the purpose of setting a seed?

In [30]:
torch.manual_seed(42)
vectorizer, model = train()
#print("f vectorizer: {vectorizer}, Model: {mode}")

--- [START] Initializing Data and Model ---
--- [STEP 1] LOADING DATA FROM reviews-train.txt ---
--- [STEP 2] LOADED 5 LINES.
splited data: {tokenized_lines}
  Raw Item 0: Label='music', Text=['oh', 'man', ',', 'this', 'sucks']...
  Raw Item 1: Label='music', Text=['his', 'singing', 'style', 'is', 'very']...
  Raw Item 2: Label='music', Text=['i', 'can', 'only', 'say', 'that']...
  Raw Item 3: Label='camera', Text=['i', 'purchased', 'this', 'camera', '2']...
  Raw Item 4: Label='camera', Text=['not', 'seeing', 'many', 'reviews', ',']...
All Data: [(['oh', 'man', ',', 'this', 'sucks', 'really', 'bad', '.', 'good', 'thing', 'nu-metal', 'is', 'dead', '.', 'thrash', 'metal', 'is', 'real', 'metal', ',', 'this', 'is', 'for', 'posers'], 'music'), (['his', 'singing', 'style', 'is', 'very', 'unique', 'as', 'are', 'his', 'lyrics', '.', 'he', 'has', 'an', 'eclectic', 'mix', 'of', 'love', 'songs', ',', '"', 'we', 'are', 'the', 'world', '"', 'type', 'songs', 'and', 'a', 'few', 'others', 'on', 'this

### Inspecting the embeddings

Now that you have trained the classifier on two separate prediction tasks, it is interesting to inspect and compare the embedding vectors it learned in the process. For this you will use an online tool called the [Embedding Projector](http://projector.tensorflow.org). The next cell contains code to save the embeddings from a trained classifier in a format that can be loaded into this tool.

In [31]:
def save_embeddings(
    vectorizer: ReviewVectorizer,
    model: Classifier,
    vectors_filename: str,
    metadata_filename: str,
):
    i2t = {i: t for t, i in vectorizer.t2i.items()}
    embeddings = model.embedding.weight.detach().numpy()
    items = [(i2t[i], embeddings[i]) for i in range(len(i2t))]
    with open(vectors_filename, "wt") as f1, open(metadata_filename, "wt") as f2:
        for w, e in items:
            print("\t".join("{:.5f}".format(x) for x in e), file=f1)
            print(w, file=f2)

Call this code as follows:

In [32]:
save_embeddings(vectorizer, model, "vectors.tsv", "metadata.tsv")

#### 🎓 Task 1.11: Inspecting the embeddings

Load the embeddings from the two classification tasks (product category classification and sentiment classification) into the Embedding Projector web app and inspect the vector spaces. How do they compare visually? Does the visualisation make sense to you?

The Embedding Projector offers visualisations based on three dimensionality reduction methods: [UMAP](https://umap-learn.readthedocs.io/en/latest/), [T-SNE](https://en.m.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding), and [PCA](https://en.m.wikipedia.org/wiki/Principal_component_analysis). Which of these seems most useful to you?

Focus on the embeddings for the words *repair* and *sturdy*. Are they close to each other or far away from another? What happens if you switch to the other task? How do you explain that?

### Initialisation of embedding layers

The error surfaces explored when training neural networks can be very complex. Because of this, it is crucial to choose “good” initial values for the parameters. In the final task of this lab, you will run a small experiment to see how alternative initialisations can affect a model’s performance.

In PyTorch, the weights of the embedding layer are initially set by sampling from the standard normal distribution, $\mathcal{N}(0, 1)$. However, research suggests other approaches may work better. For example, given that embedding layers share similarities with linear layers, it makes sense to use the same initialisation method for both. The default initialisation method for linear layers in PyTorch is the so-called Kaiming initialisation, introduced by [He et al. (2015)](https://www.cv-foundation.org/openaccess/content_iccv_2015/papers/He_Delving_Deep_into_ICCV_2015_paper.pdf).

#### 🧩 Task 1.12: Initialisation of embedding layers

Check the [source code of `nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear) to see how PyTorch initialises the weights of linear layers using the Kaiming initialisation method. Apply the same method to the embedding layer of your classifier and see how this affects the loss of your model and the vector spaces.

**🥳 Congratulations on finishing lab&nbsp;1!**